# 3/3 — Train a Policy on the Ensemble Signal

**Run after notebook 2**, and only if its diagnostics looked better than the
previous model.

Reward comes entirely from the time2success ensemble — MetaWorld's own
reward is discarded. Two reward forms are provided; pick one in Section 4.

**Steps:** Sections 1–3 set up. Section 4 chooses the reward form and checks
its magnitude. Section 5 is a 5k smoke test — read `ep_len_mean` before
committing to Section 6's full run.

Stall detection is disabled by default. With `stall_window=20` it was firing
around frame 46, cutting every episode off before the insertion could finish
and starving the replay buffer of the second half of the task.

## 1. Setup

In [3]:
import numpy as np
import torch
import torch.nn as nn
import gymnasium as gym
import metaworld
import imageio, collections, json, os, time
from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv, VecMonitor
from stable_baselines3.common.callbacks import BaseCallback

In [4]:
# ============================================================
# SHARED FEATURE BUILDER — identical copy in all three notebooks.
# If you change it here, change it EVERYWHERE. A mismatch in column
# order fails silently: the model receives numbers in the wrong slots
# and returns plausible-looking garbage without raising.
# ============================================================
HAND_POS_IDX = [0, 1, 2]
GRIPPER_IDX  = 3
PEG_POS_IDX  = [4, 5, 6]
GOAL_POS_IDX = [36, 37, 38]
GRIPPER_CLOSED_THRESHOLD = 0.6
FEATURE_DIM = 49

def build_features(obs):
    """39-dim raw obs -> 49-dim feature vector.

    Columns:
       0-38  raw obs (unchanged)
      39-41  hand->peg vector
      42-44  peg->goal vector
         45  |hand->peg|
         46  |peg->goal|
         47  gripper openness
         48  holding flag (0/1)
    """
    obs = np.asarray(obs, dtype=np.float32)
    hand = obs[HAND_POS_IDX]
    peg  = obs[PEG_POS_IDX]
    goal = obs[GOAL_POS_IDX]
    gripper = obs[GRIPPER_IDX]

    hand_to_peg_vec = peg - hand
    peg_to_goal_vec = goal - peg
    hand_to_peg = np.linalg.norm(hand_to_peg_vec)
    peg_to_goal = np.linalg.norm(peg_to_goal_vec)
    holding = float(gripper < GRIPPER_CLOSED_THRESHOLD and hand_to_peg < 0.06)

    return np.concatenate([
        obs,
        hand_to_peg_vec,
        peg_to_goal_vec,
        [hand_to_peg, peg_to_goal, gripper, holding],
    ]).astype(np.float32)

In [5]:
TASK_NAME = "peg-insert-side-v3"
SUCCESS_KEY = "success"

def make_env(seed=0, render_mode=None):
    return gym.make("Meta-World/MT1", env_name=TASK_NAME, seed=seed, render_mode=render_mode)

_probe = make_env(seed=0)
DT = getattr(_probe.unwrapped, "dt", _probe.unwrapped.model.opt.timestep)
RAW_OBS_DIM = _probe.observation_space.shape[0]
_probe.close()
print(f"DT={DT}, raw obs dim={RAW_OBS_DIM}, feature dim={FEATURE_DIM}")

DT=0.0125, raw obs dim=39, feature dim=49


d:\Miniconda3\envs\duke_rob\lib\site-packages\gymnasium\utils\passive_env_checker.py:34: UserWarning: WARN: A Box observation space maximum and minimum values are equal.
  logger.warn("A Box observation space maximum and minimum values are equal.")


In [6]:
class Time2SuccessModel(nn.Module):
    """dropout=0.2 must match whatever the saved checkpoints were trained with,
    or load_state_dict fails on layer-index mismatch."""
    def __init__(self, obs_dim, hidden=256, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cpu


## 2. Load the frozen ensemble

This is the only place the reward model enters. It stays in `eval()` and never receives gradients during RL.

In [8]:
ENSEMBLE_DIR = "./checkpoints/time2success_ensemble_v5"
N_ENSEMBLE = 5

norm = np.load(os.path.join(ENSEMBLE_DIR, "normalization.npz"))
X_MEAN, X_STD = norm["X_mean"], norm["X_std"]
Y_MEAN, Y_STD = norm["y_mean"].item(), norm["y_std"].item()
assert X_MEAN.shape[0] == FEATURE_DIM, (
    f"normalization is {X_MEAN.shape[0]}-dim, expected {FEATURE_DIM} — wrong ensemble dir?")

ensemble = []
for i in range(N_ENSEMBLE):
    m = Time2SuccessModel(obs_dim=FEATURE_DIM).to(device)
    m.load_state_dict(torch.load(os.path.join(ENSEMBLE_DIR, f"model_{i}.pt")))
    m.eval()
    ensemble.append(m)
print(f"loaded {len(ensemble)} members from {ENSEMBLE_DIR}")

loaded 5 members from ./checkpoints/time2success_ensemble_v5


## 3. Reward wrapper

`reward_mode` selects between:

- `"difference"` — `γ·Φ(s') − Φ(s)`. Pays for *improvement*. Potential-based,
  so it preserves the underlying optimal policy, and only needs the model to
  order states correctly. Standing still pays zero, which is why freezing can
  become attractive.
- `"absolute"` — `Φ(s')`. Pays for *being close*. Standing still costs
  `−pred_steps` every step, so not finishing is expensive. Loses the
  potential-based guarantee and depends on the model's absolute calibration,
  which is weaker than its ordering.

In [16]:
GAMMA = 0.99
GAMMA_SHAPING = 1
class EnsembleT2SRewardWrapper(gym.Wrapper):
    def __init__(self, env, models, device, x_mean, x_std, y_mean, y_std,
                 gamma=GAMMA_SHAPING, shaping_scale=1.0, uncertainty_weight=0.5,
                 max_pred_steps=500, reward_mode="difference", object_motion_weight = 5.0, stall_window=None):
        super().__init__(env)
        self.models = models
        self.device = device
        self.x_mean, self.x_std = x_mean, x_std
        self.y_mean, self.y_std = y_mean, y_std
        self.gamma = gamma
        self.shaping_scale = shaping_scale
        self.uncertainty_weight = uncertainty_weight
        self.max_pred_steps = max_pred_steps
        self.reward_mode = reward_mode
        self.stall_window = stall_window     # None = disabled
        self._last_phi = 0.0
        self._recent = collections.deque(maxlen=stall_window) if stall_window else None
        self.object_motion_weight = object_motion_weight   # new constructor arg, try 5.0
        self._last_peg = None

    @torch.no_grad()
    def _potential(self, obs):
        feat = build_features(obs)
        x = torch.tensor((feat - self.x_mean) / self.x_std,
                          dtype=torch.float32).unsqueeze(0).to(self.device)
        preds = np.array([m(x).item() for m in self.models]) * self.y_std + self.y_mean
        penalized = preds.mean() + self.uncertainty_weight * preds.std()
        return -float(np.clip(penalized, 0, self.max_pred_steps))

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self._last_peg = obs[PEG_POS_IDX].copy()
        self._last_phi = self._potential(obs)
        if self._recent is not None:
            self._recent.clear()
            self._recent.append(-self._last_phi)
        return obs, info

    def step(self, action):
        obs, _env_reward, terminated, truncated, info = self.env.step(action)  # env reward DISCARDED
        phi_next = self._potential(obs)

        if self.reward_mode == "difference":
            r = self.gamma * phi_next - self._last_phi
        elif self.reward_mode == "absolute":
            r = phi_next
        else:
            raise ValueError(self.reward_mode)
        
        peg_now = obs[PEG_POS_IDX]
        object_motion = float(np.linalg.norm(peg_now - self._last_peg))
        self._last_peg = peg_now.copy()

        r += self.object_motion_weight * object_motion

        if self._recent is not None:
            pred_now = -phi_next
            if len(self._recent) == self._recent.maxlen and pred_now >= max(self._recent):
                truncated = True
            self._recent.append(pred_now)

        self._last_phi = phi_next
        return obs, self.shaping_scale * r, terminated, truncated, info

## 4. Choose the reward form and check its magnitude

The two modes produce very different scales — `difference` gives fractions
per step, `absolute` gives tens. `shaping_scale` needs adjusting to match.
Run this before training; it takes seconds and catches a badly-scaled reward
that would otherwise only show up hours in.

In [22]:
REWARD_MODE = "difference"      # or "absolute"
UNCERTAINTY_WEIGHT = 0.5        # std ~1-2 on well-covered states, ~10+ on poor ones
SHAPING_SCALE = 1.0             # for "absolute", try ~0.05 to keep gradients comparable
OBJECT_MOTION_WEIGHT = 10   # adjust this based on the parity weight calculation

def make_reward_env(seed=0):
    return EnsembleT2SRewardWrapper(
        make_env(seed=seed), ensemble, device, X_MEAN, X_STD, Y_MEAN, Y_STD,
        shaping_scale=SHAPING_SCALE, uncertainty_weight=UNCERTAINTY_WEIGHT,
        reward_mode=REWARD_MODE, stall_window=None, object_motion_weight=OBJECT_MOTION_WEIGHT)

_e = make_reward_env(seed=0)
_o, _ = _e.reset()
_rs = []
for _ in range(50):
    _o, _r, _t, _tr, _ = _e.step(_e.action_space.sample())
    _rs.append(_r)
    if _t or _tr: break
_e.close()
_rs = np.array(_rs)
print(f"mode={REWARD_MODE}, scale={SHAPING_SCALE}")
print(f"random-action reward over {len(_rs)} steps: "
      f"mean={_rs.mean():.4f}, min={_rs.min():.4f}, max={_rs.max():.4f}")

mode=difference, scale=1.0
random-action reward over 50 steps: mean=-0.1128, min=-0.8731, max=0.7846


In [23]:
_e = make_reward_env(seed=0)
_o, _ = _e.reset()
_last = _o[PEG_POS_IDX].copy()
_rs, _ms = [], []
for _ in range(50):
    _o, _r, _t, _tr, _ = _e.step(_e.action_space.sample())
    _rs.append(_r)
    _ms.append(np.linalg.norm(_o[PEG_POS_IDX] - _last))
    _last = _o[PEG_POS_IDX].copy()
    if _t or _tr: break
_e.close()
_rs, _ms = np.array(_rs), np.array(_ms)
motion_contrib = OBJECT_MOTION_WEIGHT * _ms.mean()
print(f"combined reward: mean={_rs.mean():.4f}")
print(f"  of which motion: {motion_contrib:.4f} ({100*motion_contrib/max(abs(_rs.mean()),1e-9):.1f}%)")

combined reward: mean=-0.1128
  of which motion: 0.0019 (1.7%)


## 5. Training setup and smoke test

In [24]:
N_ENVS = 6
CKPT_DIR = f"./checkpoints/peg_pure_t2s_v5_{REWARD_MODE}"

train_env = VecMonitor(DummyVecEnv([lambda i=i: make_reward_env(seed=i) for i in range(N_ENVS)]))
eval_env = make_env(seed=1000)     # raw env — real success flag, never the shaped reward

policy = SAC("MlpPolicy", env=train_env, learning_rate=3e-4, buffer_size=1_000_000,
             batch_size=256, tau=0.005, gamma=GAMMA, ent_coef="auto",
             policy_kwargs=dict(net_arch=[400, 400]),
             tensorboard_log=f"./tb_logs/peg_pure_t2s_v5_{REWARD_MODE}",
             verbose=1, seed=0)

class SuccessCallback(BaseCallback):
    def __init__(self, eval_env, eval_freq=25_000, n_eval_episodes=10,
                 ckpt_dir=CKPT_DIR, ckpt_freq=500_000, verbose=1):
        super().__init__(verbose)
        self.eval_env, self.eval_freq = eval_env, eval_freq
        self.n_eval_episodes = n_eval_episodes
        self.ckpt_dir, self.ckpt_freq = ckpt_dir, ckpt_freq
        os.makedirs(ckpt_dir, exist_ok=True)
        self.history = []

    def _eval_once(self):
        obs, _ = self.eval_env.reset()
        for t in range(500):
            a, _ = self.model.predict(obs, deterministic=True)
            obs, _, term, trunc, info = self.eval_env.step(a)
            if info.get(SUCCESS_KEY, 0):
                return t
            if term or trunc:
                break
        return None

    def _on_step(self):
        if self.num_timesteps % self.ckpt_freq < self.training_env.num_envs:
            self.model.save(os.path.join(self.ckpt_dir, f"ckpt_{self.num_timesteps}.zip"))
        if self.num_timesteps % self.eval_freq < self.training_env.num_envs:
            steps = [self._eval_once() for _ in range(self.n_eval_episodes)]
            sr = sum(s is not None for s in steps) / self.n_eval_episodes
            times = [s for s in steps if s is not None]
            mt = float(np.mean(times)) if times else None
            self.logger.record("eval/success_rate", sr)
            if mt is not None:
                self.logger.record("eval/mean_time_to_success", mt)
            self.history.append(dict(step=self.num_timesteps, success_rate=sr,
                                       mean_time_to_success=mt, timestamp=time.time()))
            with open(os.path.join(self.ckpt_dir, "eval_history.json"), "w") as f:
                json.dump(self.history, f, indent=2)
            print(f"[{time.strftime('%H:%M:%S')}] eval @ {self.num_timesteps}: "
                  f"success_rate={sr:.2f} mean_t2s={mt}")
        return True

callback = SuccessCallback(eval_env=eval_env, ckpt_freq=2000, eval_freq=2000)

Using cpu device


In [25]:
_t0 = time.time()
policy.learn(total_timesteps=5_000, callback=callback, tb_log_name="smoke")
print(f"\n5,000 steps in {time.time()-_t0:.0f}s — "
      f"3M would take ~{(time.time()-_t0)*600/3600:.1f} hours")
print("Check rollout/ep_len_mean in the output above: should be near 500. "
      "If it is ~46, stall detection is somehow still active.")

Logging to ./tb_logs/peg_pure_t2s_v5_difference\smoke_3
[20:26:36] eval @ 2004: success_rate=0.00 mean_t2s=None
---------------------------------
| eval/              |          |
|    success_rate    | 0        |
| rollout/           |          |
|    ep_len_mean     | 500      |
|    ep_rew_mean     | 3.37     |
| time/              |          |
|    episodes        | 4        |
|    fps             | 153      |
|    time_elapsed    | 19       |
|    total_timesteps | 3000     |
| train/             |          |
|    actor_loss      | -8.89    |
|    critic_loss     | 0.358    |
|    ent_coef        | 0.865    |
|    ent_coef_loss   | -0.977   |
|    learning_rate   | 0.0003   |
|    n_updates       | 483      |
---------------------------------
[20:26:51] eval @ 4002: success_rate=0.00 mean_t2s=None

5,000 steps in 33s — 3M would take ~5.5 hours
Check rollout/ep_len_mean in the output above: should be near 500. If it is ~46, stall detection is somehow still active.


In [26]:
_e = make_reward_env(seed=0)
_o, _ = _e.reset()
_last = _o[PEG_POS_IDX].copy()
temporal_sum, motion_sum = 0.0, 0.0
for _ in range(500):
    _o, _r, _t, _tr, _ = _e.step(_e.action_space.sample())
    m = np.linalg.norm(_o[PEG_POS_IDX] - _last) * OBJECT_MOTION_WEIGHT
    motion_sum += m
    temporal_sum += _r - m
    _last = _o[PEG_POS_IDX].copy()
    if _t or _tr: break
_e.close()
print(f"episode return: temporal={temporal_sum:.1f}, motion={motion_sum:.1f}")

episode return: temporal=-18.5, motion=0.1


In [27]:
_e = make_reward_env(seed=0)
_o, _ = _e.reset()
phis, rs = [_e._potential(_o)], []
for _ in range(500):
    _o, _r, _t, _tr, _ = _e.step(_e.action_space.sample())
    phis.append(_e._potential(_o)); rs.append(_r)
    if _t or _tr: break
_e.close()
phis = np.array(phis)
print(f"phi range: {phis.min():.1f} to {phis.max():.1f}")
print(f"telescoped (phi_end - phi_start): {phis[-1] - phis[0]:.1f}")
print(f"actual sum of rewards: {np.sum(rs):.1f}")
print(f"gamma residual estimate: {-(1-GAMMA) * phis[:-1].sum():.1f}")
print(f"clipped at 0: {(phis == 0).sum()}, clipped at -500: {(phis == -500).sum()}")

phi range: -98.3 to -56.0
telescoped (phi_end - phi_start): -18.5
actual sum of rewards: -18.4
gamma residual estimate: 418.5
clipped at 0: 0, clipped at -500: 0


## 6. Full run

In [28]:
TOTAL_TIMESTEPS = 3_000_000
callback = SuccessCallback(eval_env=eval_env, ckpt_freq=500_000, eval_freq=25_000)

print(f"[{time.strftime('%H:%M:%S')}] starting learn()")
policy.learn(total_timesteps=TOTAL_TIMESTEPS, callback=callback,
             tb_log_name="run1", progress_bar=False, reset_num_timesteps=False)
print(f"[{time.strftime('%H:%M:%S')}] learn() RETURNED")

policy.save(os.path.join(CKPT_DIR, "final"))
print(f"[{time.strftime('%H:%M:%S')}] MODEL SAVED — safe to interrupt from here")

for name, e in [("eval", eval_env), ("train", train_env)]:
    try:
        e.close()
        print(f"[{time.strftime('%H:%M:%S')}] {name} env closed")
    except Exception as ex:
        print(f"{name} env close failed:", ex)
print(f"[{time.strftime('%H:%M:%S')}] CELL COMPLETE")

[20:30:03] starting learn()
Logging to ./tb_logs/peg_pure_t2s_v5_difference\run1_0
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 500      |
|    ep_rew_mean     | 10.6     |
| time/              |          |
|    episodes        | 8        |
|    fps             | 237      |
|    time_elapsed    | 4        |
|    total_timesteps | 6000     |
| train/             |          |
|    actor_loss      | -12.8    |
|    critic_loss     | 0.289    |
|    ent_coef        | 0.745    |
|    ent_coef_loss   | -1.98    |
|    learning_rate   | 0.0003   |
|    n_updates       | 983      |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 500      |
|    ep_rew_mean     | 10.6     |
| time/              |          |
|    episodes        | 12       |
|    fps             | 237      |
|    time_elapsed    | 4        |
|    total_timesteps | 6000     |
---------------------------------

If it hangs, the last printed line localizes it. `MODEL SAVED` with no
`train env closed` is MuJoCo cleanup blocking — harmless, the model is
already on disk. No `learn() RETURNED` means the stall is inside training.

## 7. Evaluate the resulting policy

In [29]:
eval_policy = SAC.load(os.path.join(CKPT_DIR, "final"))

steps = []
for seed in range(20):
    e = make_env(seed=seed)
    obs, _ = e.reset()
    ss = None
    for t in range(500):
        a, _ = eval_policy.predict(obs, deterministic=True)
        obs, _, term, trunc, info = e.step(a)
        if info.get(SUCCESS_KEY, 0):
            ss = t; break
        if term or trunc: break
    e.close()
    steps.append(ss)

sr = np.mean([s is not None for s in steps])
times = [s for s in steps if s is not None]
print(f"success rate over 20 seeds: {sr:.0%}")
if times:
    print(f"mean time-to-success: {np.mean(times):.1f} steps")

success rate over 20 seeds: 0%


In [30]:
e = make_env(seed=0, render_mode="rgb_array")
obs, _ = e.reset()
frames, ss = [], None
for t in range(500):
    frames.append(e.render())
    a, _ = eval_policy.predict(obs, deterministic=True)
    obs, _, term, trunc, info = e.step(a)
    if info.get(SUCCESS_KEY, 0) and ss is None:
        ss = t
    if term or trunc: break
e.close()
imageio.mimsave("policy_v5.mp4", frames, fps=20)
print(f"saved policy_v5.mp4, success_step={ss}, frames={len(frames)}")

saved policy_v5.mp4, success_step=None, frames=500


In [ ]:
from IPython.display import Video
Video("policy_v5.mp4", embed=True)

## 8. Compare against stage 1

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

with open("./checkpoints/peg_insert_side/eval_history.json") as f:
    orig = pd.DataFrame(json.load(f))
with open(os.path.join(CKPT_DIR, "eval_history.json")) as f:
    new = pd.DataFrame(json.load(f))

plt.figure(figsize=(7, 4))
plt.plot(orig["step"], orig["success_rate"], label="stage 1 (MetaWorld dense reward)")
plt.plot(new["step"], new["success_rate"], label=f"time2success only ({REWARD_MODE})")
plt.xlabel("step"); plt.ylabel("success rate"); plt.legend()
plt.title("Dense reward vs. pure time2success signal")
plt.show()